# 08 - Data Merging and Alignment

## Objective
Merge all prepared datasets into a single master dataset for machine learning modeling, ensuring:
- Correct time alignment
- Leakage prevention (shift external features by 1 day)
- Dataset integrity

## Input Datasets
1. **Stock + Technical Indicators**: `Market_Data/features/stock_with_indicators.csv`
2. **Global Market Data**: `Market_Data/raw/global_market_data.csv`
3. **GDELT Events**: `Market_Data/raw/gdelt_events.csv`
4. **Macro Economic Data**: `Market_Data/raw/macro_data.csv`

**Note**: News sentiment data is excluded due to incomplete coverage.

## Output
- `Market_Data/processed/merged_dataset.parquet`

## Setup

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# Set display options
pd.set_option('display.max_columns', 50)
pd.set_option('display.width', None)

# Data paths
DATA_DIR = Path('../Market_Data')
RAW_DIR = DATA_DIR / 'raw'
FEATURES_DIR = DATA_DIR / 'features'
PROCESSED_DIR = DATA_DIR / 'processed'

# Create processed directory if not exists
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

print('Setup complete.')

Setup complete.


---
## Step 1: Load All Datasets

In [2]:
# Load Stock + Technical Indicators (base table)
stock_df = pd.read_csv(FEATURES_DIR / 'stock_with_indicators.csv')
stock_df['Date'] = pd.to_datetime(stock_df['Date'])
stock_df = stock_df.sort_values(['Ticker', 'Date']).reset_index(drop=True)

print('Stock + Technical Indicators loaded.')
print(f'Shape: {stock_df.shape}')
print(f'Date range: {stock_df["Date"].min()} to {stock_df["Date"].max()}')
print(f'Unique tickers: {stock_df["Ticker"].nunique()}')
print(f'Columns: {list(stock_df.columns)}')

Stock + Technical Indicators loaded.
Shape: (65895, 22)
Date range: 2023-03-15 00:00:00 to 2025-12-31 00:00:00
Unique tickers: 100
Columns: ['Date', 'Ticker', 'Open', 'High', 'Low', 'Close', 'Volume', 'Return', 'Log_Return', 'RSI', 'ROC', 'EMA_20', 'SMA_20', 'MACD', 'MACD_Signal', 'BB_upper', 'BB_lower', 'ATR', 'Volatility_20', 'Volatility_50', 'Volume_MA_20', 'OBV']


In [3]:
# Load Global Market Data
global_df = pd.read_csv(RAW_DIR / 'global_market_data.csv')
global_df['Date'] = pd.to_datetime(global_df['Date'])
global_df = global_df.sort_values('Date').reset_index(drop=True)

print('Global Market Data loaded.')
print(f'Shape: {global_df.shape}')
print(f'Date range: {global_df["Date"].min()} to {global_df["Date"].max()}')
print(f'Columns: {list(global_df.columns)}')

Global Market Data loaded.
Shape: (751, 9)
Date range: 2023-01-04 00:00:00 to 2025-12-31 00:00:00
Columns: ['Date', 'SP500_RET', 'NASDAQ_RET', 'DOW_RET', 'GOLD_RET', 'OIL_RET', 'USDINR_RET', 'VIX_RET', 'NIFTY_RET']


In [4]:
# Load GDELT Events Data
gdelt_df = pd.read_csv(RAW_DIR / 'gdelt_events.csv')
gdelt_df['Date'] = pd.to_datetime(gdelt_df['Date'])
gdelt_df = gdelt_df.sort_values('Date').reset_index(drop=True)

print('GDELT Events Data loaded.')
print(f'Shape: {gdelt_df.shape}')
print(f'Date range: {gdelt_df["Date"].min()} to {gdelt_df["Date"].max()}')
print(f'Columns: {list(gdelt_df.columns)}')

GDELT Events Data loaded.
Shape: (1097, 8)
Date range: 2023-01-01 00:00:00 to 2026-01-01 00:00:00
Columns: ['Date', 'Event_Count', 'Avg_Tone', 'War_Flag', 'Crisis_Flag', 'Inflation_Flag', 'Rate_Hike_Flag', 'Recession_Flag']


In [5]:
# Load Macro Economic Data
macro_df = pd.read_csv(RAW_DIR / 'macro_data.csv')
macro_df['Date'] = pd.to_datetime(macro_df['Date'])
macro_df = macro_df.sort_values('Date').reset_index(drop=True)

print('Macro Economic Data loaded.')
print(f'Shape: {macro_df.shape}')
print(f'Date range: {macro_df["Date"].min()} to {macro_df["Date"].max()}')
print(f'Columns: {list(macro_df.columns)}')

Macro Economic Data loaded.
Shape: (1097, 5)
Date range: 2023-01-01 00:00:00 to 2026-01-01 00:00:00
Columns: ['Date', 'Interest_Rate', 'Inflation', 'Unemployment', 'GDP']


### Store Original Row Count for Validation

In [6]:
original_stock_rows = len(stock_df)
original_tickers = stock_df['Ticker'].nunique()
print(f'Original stock dataset: {original_stock_rows} rows, {original_tickers} tickers')

Original stock dataset: 65895 rows, 100 tickers


---
## Step 2: Shift External Features to Prevent Data Leakage

To ensure the model uses only information available at time `t` to predict `t+1`:
- Global market returns are shifted by 1 day
- GDELT event features are shifted by 1 day
- Macro variables are shifted by 1 day for strict causality

In [7]:
# Define columns to shift for each external dataset
global_columns = ['SP500_RET', 'NASDAQ_RET', 'DOW_RET', 'GOLD_RET', 'OIL_RET', 'USDINR_RET', 'VIX_RET', 'NIFTY_RET']
gdelt_columns = ['Event_Count', 'Avg_Tone', 'War_Flag', 'Crisis_Flag', 'Inflation_Flag', 'Rate_Hike_Flag', 'Recession_Flag']
macro_columns = ['Interest_Rate', 'Inflation', 'Unemployment', 'GDP']

print('Columns to shift defined.')

Columns to shift defined.


In [8]:
# Shift Global Market Data by 1 day
global_df_shifted = global_df.copy()
global_df_shifted[global_columns] = global_df_shifted[global_columns].shift(1)

print('Global market data shifted by 1 day.')
print(f'First few rows after shift:')
global_df_shifted.head()

Global market data shifted by 1 day.
First few rows after shift:


,Date,SP500_RET,NASDAQ_RET,DOW_RET,GOLD_RET,OIL_RET,USDINR_RET,VIX_RET,NIFTY_RET
0,2023-01-04,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2023-01-05,0.007539,0.006911,0.004026,0.007121,-0.053165,0.000964,-0.038865,-0.010399
2,2023-01-06,-0.011646,-0.014679,-0.010210,-0.009715,0.011395,-0.001440,0.020445,-0.002815
3,2023-01-09,0.022841,0.025623,0.021273,0.016023,0.001357,-0.000258,-0.059216,-0.007376
4,2023-01-10,-0.000768,0.006279,-0.003359,0.004560,0.011658,-0.004499,0.039754,0.013536


In [9]:
# Shift GDELT Event Data by 1 day
gdelt_df_shifted = gdelt_df.copy()
gdelt_df_shifted[gdelt_columns] = gdelt_df_shifted[gdelt_columns].shift(1)

print('GDELT event data shifted by 1 day.')
print(f'First few rows after shift:')
gdelt_df_shifted.head()

GDELT event data shifted by 1 day.
First few rows after shift:


,Date,Event_Count,Avg_Tone,War_Flag,Crisis_Flag,Inflation_Flag,Rate_Hike_Flag,Recession_Flag
0,2023-01-01,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2023-01-02,790.0,0.1398,0.0,0.0,0.0,0.0,0.0
2,2023-01-03,1226.0,-1.0132,0.0,0.0,0.0,0.0,0.0
3,2023-01-04,1155.0,-1.5468,0.0,0.0,0.0,0.0,0.0
4,2023-01-05,1210.0,-2.4713,0.0,0.0,0.0,0.0,1.0


In [10]:
# Shift Macro Economic Data by 1 day
macro_df_shifted = macro_df.copy()
macro_df_shifted[macro_columns] = macro_df_shifted[macro_columns].shift(1)

print('Macro economic data shifted by 1 day.')
print(f'First few rows after shift:')
macro_df_shifted.head()

Macro economic data shifted by 1 day.
First few rows after shift:


,Date,Interest_Rate,Inflation,Unemployment,GDP
0,2023-01-01,NaN,NaN,NaN,NaN
1,2023-01-02,4.33,300.42,3.5,27216.445
2,2023-01-03,4.33,300.42,3.5,27216.445
3,2023-01-04,4.33,300.42,3.5,27216.445
4,2023-01-05,4.33,300.42,3.5,27216.445


---
## Step 3: Merge Datasets

Merge order:
1. stock_with_indicators (base) + global_market_data
2. result + gdelt_events
3. result + macro_data

Using **LEFT JOIN** to preserve all stock rows.

In [11]:
# Merge 1: Stock + Global Market Data
merged_df = stock_df.merge(
    global_df_shifted,
    on='Date',
    how='left'
)

print('Merge 1: Stock + Global Market Data')
print(f'Shape after merge: {merged_df.shape}')
print(f'Rows preserved: {len(merged_df) == original_stock_rows}')

Merge 1: Stock + Global Market Data
Shape after merge: (65895, 30)
Rows preserved: True


In [12]:
# Merge 2: Add GDELT Events
merged_df = merged_df.merge(
    gdelt_df_shifted,
    on='Date',
    how='left'
)

print('Merge 2: + GDELT Events')
print(f'Shape after merge: {merged_df.shape}')
print(f'Rows preserved: {len(merged_df) == original_stock_rows}')

Merge 2: + GDELT Events
Shape after merge: (65895, 37)
Rows preserved: True


In [13]:
# Merge 3: Add Macro Economic Data
merged_df = merged_df.merge(
    macro_df_shifted,
    on='Date',
    how='left'
)

print('Merge 3: + Macro Economic Data')
print(f'Shape after merge: {merged_df.shape}')
print(f'Rows preserved: {len(merged_df) == original_stock_rows}')

Merge 3: + Macro Economic Data
Shape after merge: (65895, 41)
Rows preserved: True


---
## Step 4: Data Integrity Checks

In [14]:
# Check for duplicate (Date, Ticker) pairs
duplicates = merged_df.duplicated(subset=['Date', 'Ticker'], keep=False)
duplicate_count = duplicates.sum()

print(f'Duplicate (Date, Ticker) pairs: {duplicate_count}')
if duplicate_count > 0:
    print('WARNING: Duplicates found!')
    print(merged_df[duplicates].head(10))
else:
    print('No duplicates found.')

Duplicate (Date, Ticker) pairs: 0
No duplicates found.


In [15]:
# Check number of unique tickers
unique_tickers = merged_df['Ticker'].nunique()
print(f'Number of unique tickers: {unique_tickers}')
print(f'Tickers preserved: {unique_tickers == original_tickers}')

Number of unique tickers: 100
Tickers preserved: True


In [16]:
# Check date range coverage
print(f'Date range: {merged_df["Date"].min()} to {merged_df["Date"].max()}')
print(f'Number of unique dates: {merged_df["Date"].nunique()}')

Date range: 2023-03-15 00:00:00 to 2025-12-31 00:00:00
Number of unique dates: 690


In [17]:
# Check missing values percentage per column
missing_pct = (merged_df.isnull().sum() / len(merged_df) * 100).round(2)
missing_summary = pd.DataFrame({
    'Missing Count': merged_df.isnull().sum(),
    'Missing %': missing_pct
})
print('Missing Values Summary (Before Filling):')
print(missing_summary[missing_summary['Missing Count'] > 0])

Missing Values Summary (Before Filling):
            Missing Count  Missing %
SP500_RET            2198       3.34
NASDAQ_RET           2198       3.34
DOW_RET              2198       3.34
GOLD_RET             2198       3.34
OIL_RET              2198       3.34
USDINR_RET           2198       3.34
VIX_RET              2198       3.34
NIFTY_RET            2198       3.34


In [18]:
# Verify stock rows were not dropped
print(f'Original stock rows: {original_stock_rows}')
print(f'Merged dataset rows: {len(merged_df)}')
print(f'All stock rows preserved: {len(merged_df) == original_stock_rows}')

Original stock rows: 65895
Merged dataset rows: 65895
All stock rows preserved: True


---
## Step 5: Handle Missing Values

In [19]:
# Forward fill global market missing values
for col in global_columns:
    if col in merged_df.columns:
        merged_df[col] = merged_df[col].ffill()

print('Global market features forward filled.')

Global market features forward filled.


In [20]:
# Fill GDELT flags with 0
gdelt_flag_columns = ['War_Flag', 'Crisis_Flag', 'Inflation_Flag', 'Rate_Hike_Flag', 'Recession_Flag']
for col in gdelt_flag_columns:
    if col in merged_df.columns:
        merged_df[col] = merged_df[col].fillna(0).astype(int)

# Fill Event_Count and Avg_Tone with 0
if 'Event_Count' in merged_df.columns:
    merged_df['Event_Count'] = merged_df['Event_Count'].fillna(0)
if 'Avg_Tone' in merged_df.columns:
    merged_df['Avg_Tone'] = merged_df['Avg_Tone'].fillna(0)

print('GDELT features filled with 0.')

GDELT features filled with 0.


In [21]:
# Forward fill macro economic variables
for col in macro_columns:
    if col in merged_df.columns:
        merged_df[col] = merged_df[col].ffill()

print('Macro economic features forward filled.')

Macro economic features forward filled.


In [22]:
# Check remaining missing values
missing_after = merged_df.isnull().sum()
missing_after_summary = missing_after[missing_after > 0]

print('Missing Values After Filling:')
if len(missing_after_summary) > 0:
    print(missing_after_summary)
else:
    print('No missing values in external features.')

# Show total missing by column category
stock_cols = ['Open', 'High', 'Low', 'Close', 'Volume', 'Return', 'Log_Return']
tech_cols = ['RSI', 'ROC', 'EMA_20', 'SMA_20', 'MACD', 'MACD_Signal', 'BB_upper', 'BB_lower', 'ATR', 'Volatility_20', 'Volatility_50', 'Volume_MA_20', 'OBV']

print(f'\nStock price missing: {merged_df[stock_cols].isnull().sum().sum()}')
print(f'Technical indicator missing: {merged_df[tech_cols].isnull().sum().sum()}')

Missing Values After Filling:
No missing values in external features.

Stock price missing: 0
Technical indicator missing: 0


---
## Step 6: Final Dataset Structure

In [23]:
# Organize columns in logical order
id_cols = ['Date', 'Ticker']
price_cols = ['Open', 'High', 'Low', 'Close', 'Volume']
return_cols = ['Return', 'Log_Return']
tech_indicator_cols = ['RSI', 'ROC', 'EMA_20', 'SMA_20', 'MACD', 'MACD_Signal', 'BB_upper', 'BB_lower', 'ATR', 'Volatility_20', 'Volatility_50', 'Volume_MA_20', 'OBV']
global_market_cols = ['SP500_RET', 'NASDAQ_RET', 'DOW_RET', 'GOLD_RET', 'OIL_RET', 'USDINR_RET', 'VIX_RET', 'NIFTY_RET']
gdelt_cols = ['Event_Count', 'Avg_Tone', 'War_Flag', 'Crisis_Flag', 'Inflation_Flag', 'Rate_Hike_Flag', 'Recession_Flag']
macro_cols = ['Interest_Rate', 'Inflation', 'Unemployment', 'GDP']

# Combine all column lists
final_column_order = id_cols + price_cols + return_cols + tech_indicator_cols + global_market_cols + gdelt_cols + macro_cols

# Verify all columns exist
existing_cols = [col for col in final_column_order if col in merged_df.columns]
missing_cols = [col for col in final_column_order if col not in merged_df.columns]

if missing_cols:
    print(f'WARNING: Missing columns: {missing_cols}')
else:
    print('All expected columns present.')

# Reorder columns
merged_df = merged_df[existing_cols]

print(f'\nFinal column order ({len(existing_cols)} columns):')
for i, col in enumerate(existing_cols, 1):
    print(f'{i:2}. {col}')

All expected columns present.

Final column order (41 columns):
 1. Date
 2. Ticker
 3. Open
 4. High
 5. Low
 6. Close
 7. Volume
 8. Return
 9. Log_Return
10. RSI
11. ROC
12. EMA_20
13. SMA_20
14. MACD
15. MACD_Signal
16. BB_upper
17. BB_lower
18. ATR
19. Volatility_20
20. Volatility_50
21. Volume_MA_20
22. OBV
23. SP500_RET
24. NASDAQ_RET
25. DOW_RET
26. GOLD_RET
27. OIL_RET
28. USDINR_RET
29. VIX_RET
30. NIFTY_RET
31. Event_Count
32. Avg_Tone
33. War_Flag
34. Crisis_Flag
35. Inflation_Flag
36. Rate_Hike_Flag
37. Recession_Flag
38. Interest_Rate
39. Inflation
40. Unemployment
41. GDP


In [24]:
# Sort final dataset
merged_df = merged_df.sort_values(['Ticker', 'Date']).reset_index(drop=True)
print('Dataset sorted by Ticker and Date.')

Dataset sorted by Ticker and Date.


In [25]:
# Display sample rows
print('Sample rows from final dataset:')
merged_df.head(10)

Sample rows from final dataset:


,Date,Ticker,Open,High,Low,Close,Volume,Return,Log_Return,RSI,ROC,EMA_20,SMA_20,MACD,MACD_Signal,BB_upper,BB_lower,ATR,Volatility_20,Volatility_50,Volume_MA_20,OBV,SP500_RET,NASDAQ_RET,DOW_RET,GOLD_RET,OIL_RET,USDINR_RET,VIX_RET,NIFTY_RET,Event_Count,Avg_Tone,War_Flag,Crisis_Flag,Inflation_Flag,Rate_Hike_Flag,Recession_Flag,Interest_Rate,Inflation,Unemployment,GDP
0,2023-03-15,ABB,3344.000000,3347.949951,3286.000000,3302.250000,228737,-0.002688,-0.002691,61.089784,2.638815,3239.739084,3245.569983,88.782405,96.142584,3429.555071,3061.584895,85.322652,0.015609,0.016247,348395.80,4383288,0.016477,0.021388,0.010568,-0.002877,-0.046390,0.005605,-0.105204,-0.006471,1201.0,-3.5560,0,0,0,0,0,4.65,301.821,3.5,27216.445
1,2023-03-16,ABB,3282.050049,3344.649902,3252.000000,3327.949951,257895,0.007783,0.007752,63.064254,0.498271,3248.140119,3254.487476,85.193424,93.952752,3436.277668,3072.697283,85.846027,0.014939,0.016254,330304.45,4641183,-0.006981,0.000516,-0.008734,0.010702,-0.052152,-0.001365,0.101559,-0.004175,1215.0,1.3026,0,0,1,0,0,4.65,301.821,3.5,27216.445
2,2023-03-17,ABB,3347.050049,3364.000000,3286.600098,3312.550049,256724,-0.004627,-0.004638,61.064636,-0.326468,3254.274398,3260.694983,80.182196,91.198641,3441.510581,3079.879385,85.242732,0.014855,0.016302,330495.80,4384459,0.017562,0.024771,0.011670,-0.003945,0.010945,0.004991,-0.120505,0.000792,1221.0,-1.4995,1,0,0,0,0,4.65,301.821,3.5,27216.445
3,2023-03-20,ABB,3300.000000,3329.000000,3266.449951,3298.850098,175472,-0.004136,-0.004144,59.264343,-0.898232,3258.519703,3268.829993,74.249390,87.808791,3440.930063,3096.729923,83.621826,0.014298,0.015626,321571.10,4208987,-0.011019,-0.007405,-0.011926,0.026472,-0.023555,-0.001123,0.109613,0.006738,844.0,-1.1120,0,0,0,1,0,4.65,301.821,3.5,27216.445
4,2023-03-21,ABB,3299.000000,3432.949951,3299.000000,3412.350098,278420,0.034406,0.033827,67.747821,2.829639,3273.170216,3280.740002,77.809164,85.808866,3457.877748,3103.602257,87.227400,0.015860,0.016174,320222.80,4487407,0.008918,0.003872,0.012008,0.004772,0.013485,-0.001087,-0.053312,-0.006529,1172.0,0.5627,0,0,0,1,0,4.65,301.821,3.5,27216.445
5,2023-03-22,ABB,3413.050049,3427.199951,3301.050049,3309.899902,288396,-0.030023,-0.030483,56.341858,-2.069624,3276.668282,3287.765002,71.538782,82.954849,3457.680025,3117.849980,90.007578,0.017549,0.016859,318916.50,4199011,0.012982,0.015808,0.009801,-0.020816,0.024985,-0.000047,-0.114700,0.007011,1238.0,-3.6678,1,0,0,0,0,4.65,301.821,3.5,27216.445
6,2023-03-23,ABB,3310.000000,3384.000000,3301.350098,3360.449951,369001,0.015272,0.015157,59.926815,-0.813166,3284.647488,3297.147498,69.843315,80.332542,3461.257894,3133.037101,89.482030,0.017782,0.016880,326446.20,4568012,-0.016463,-0.016033,-0.016292,0.004541,0.022645,0.001792,0.041160,0.002595,1203.0,-3.1453,0,0,0,0,0,4.65,301.821,3.5,27216.445
7,2023-03-24,ABB,3377.350098,3392.000000,3330.000000,3346.699951,180779,-0.004092,-0.004100,58.519203,-0.908982,3290.557247,3307.564990,66.622156,77.590465,3455.706449,3159.423531,87.519028,0.017567,0.016730,327698.35,4387233,0.002985,0.010063,0.002346,0.024142,-0.013258,-0.001829,0.015723,-0.004373,1140.0,-2.2593,0,0,0,0,0,4.65,301.821,3.5,27216.445
8,2023-03-27,ABB,3354.000000,3368.899902,3313.800049,3324.600098,171779,-0.006603,-0.006625,56.232951,1.531559,3293.799423,3316.694995,61.576278,74.387627,3443.927655,3189.462335,85.203373,0.017703,0.016783,329201.40,4215454,0.005640,0.003102,0.004120,-0.005868,-0.010006,-0.003157,-0.038479,-0.007721,816.0,-2.6906,0,1,0,1,0,4.65,301.821,3.5,27216.445
9,2023-03-28,ABB,3357.000000,3385.000000,3303.000000,3338.649902,184543,0.004226,0.004217,57.373134,0.830527,3298.070897,3328.224988,58.042013,71.118505,3412.166918,3244.283057,84.974560,0.017404,0.016747,327428.50,4399997,0.001647,-0.004662,0.006035,-0.014984,0.051256,0.000893,-0.052438,0.002399,1254.0,0.6860,0,0,0,0,0,4.65,301.821,3.5,27216.445


In [26]:
# Display data types
print('Data types:')
merged_df.dtypes

Data types:


Date              datetime64[ns]
Ticker                    object
Open                     float64
High                     float64
Low                      float64
Close                    float64
Volume                     int64
Return                   float64
Log_Return               float64
RSI                      float64
ROC                      float64
EMA_20                   float64
SMA_20                   float64
MACD                     float64
MACD_Signal              float64
BB_upper                 float64
BB_lower                 float64
ATR                      float64
Volatility_20            float64
Volatility_50            float64
Volume_MA_20             float64
OBV                        int64
SP500_RET                float64
NASDAQ_RET               float64
DOW_RET                  float64
GOLD_RET                 float64
OIL_RET                  float64
USDINR_RET               float64
VIX_RET                  float64
NIFTY_RET                float64
Event_Coun

---
## Step 7: Save Output

In [36]:
# Save to Parquet format
output_path = PROCESSED_DIR / 'merged_dataset.parquet'
merged_df.to_parquet(output_path, index=False, engine='pyarrow')
## Saving it in csv format as well for easy access
merged_df.to_csv(PROCESSED_DIR / 'merged_dataset.csv', index=False)
print(f'Merged dataset saved to: {output_path}')
print(f'File size: {output_path.stat().st_size / 1024 / 1024:.2f} MB')

Merged dataset saved to: ..\Market_Data\processed\merged_dataset.parquet
File size: 11.20 MB


In [28]:
# Verify saved file
verify_df = pd.read_parquet(output_path)
print(f'Verification - Loaded shape: {verify_df.shape}')
print(f'Verification - Matches saved: {verify_df.shape == merged_df.shape}')

Verification - Loaded shape: (65895, 41)
Verification - Matches saved: True


---
## Step 8: Validation Output

In [29]:
print('=' * 60)
print('FINAL DATASET SUMMARY')
print('=' * 60)
print(f'Dataset shape: {merged_df.shape}')
print(f'Number of rows: {len(merged_df):,}')
print(f'Number of columns: {len(merged_df.columns)}')
print(f'Number of unique tickers: {merged_df["Ticker"].nunique()}')
print(f'Date range: {merged_df["Date"].min()} to {merged_df["Date"].max()}')
print(f'Number of trading days: {merged_df["Date"].nunique()}')
print('=' * 60)

FINAL DATASET SUMMARY
Dataset shape: (65895, 41)
Number of rows: 65,895
Number of columns: 41
Number of unique tickers: 100
Date range: 2023-03-15 00:00:00 to 2025-12-31 00:00:00
Number of trading days: 690


In [30]:
# Missing value summary
print('MISSING VALUE SUMMARY')
print('-' * 60)
missing_final = merged_df.isnull().sum()
missing_final_pct = (missing_final / len(merged_df) * 100).round(2)

for col in merged_df.columns:
    if missing_final[col] > 0:
        print(f'{col}: {missing_final[col]:,} ({missing_final_pct[col]}%)')

total_missing = missing_final.sum()
if total_missing == 0:
    print('No missing values in external features (Global, GDELT, Macro).')
    print('Note: Stock/Technical indicator missing values are expected and not filled.')
print('-' * 60)

MISSING VALUE SUMMARY
------------------------------------------------------------
No missing values in external features (Global, GDELT, Macro).
Note: Stock/Technical indicator missing values are expected and not filled.
------------------------------------------------------------


In [31]:
# List all tickers
print('TICKERS IN DATASET')
print('-' * 60)
tickers = sorted(merged_df['Ticker'].unique())
for i, ticker in enumerate(tickers, 1):
    print(f'{i:3}. {ticker}')

TICKERS IN DATASET
------------------------------------------------------------
  1. ABB
  2. ADANIENSOL
  3. ADANIENT
  4. ADANIGREEN
  5. ADANIPORTS
  6. ADANIPOWER
  7. AMBUJACEM
  8. APOLLOHOSP
  9. ASIANPAINT
 10. AXISBANK
 11. BAJAJ-AUTO
 12. BAJAJFINSV
 13. BAJAJHLDNG
 14. BAJFINANCE
 15. BANKBARODA
 16. BEL
 17. BHARTIARTL
 18. BOSCHLTD
 19. BPCL
 20. BRITANNIA
 21. CANBK
 22. CGPOWER
 23. CHOLAFIN
 24. CIPLA
 25. COALINDIA
 26. CUMMINSIND
 27. DIVISLAB
 28. DLF
 29. DMART
 30. DRREDDY
 31. EICHERMOT
 32. ENRIN
 33. ETERNAL
 34. GAIL
 35. GODREJCP
 36. GRASIM
 37. HAL
 38. HCLTECH
 39. HDFCAMC
 40. HDFCBANK
 41. HDFCLIFE
 42. HINDALCO
 43. HINDUNILVR
 44. HINDZINC
 45. HYUNDAI
 46. ICICIBANK
 47. INDHOTEL
 48. INDIGO
 49. INFY
 50. IRFC
 51. ITC
 52. JINDALSTEL
 53. JIOFIN
 54. JSWSTEEL
 55. KOTAKBANK
 56. LODHA
 57. LT
 58. M&M
 59. MARUTI
 60. MAXHEALTH
 61. MAZDOCK
 62. MOTHERSON
 63. MUTHOOTFIN
 64. NESTLEIND
 65. NIFTY50
 66. NTPC
 67. ONGC
 68. PAYTM
 69. PFC
 70. PIDILIT

In [32]:
# Feature categories summary
print('FEATURE CATEGORIES')
print('-' * 60)
print(f'ID columns: {len(id_cols)} - {id_cols}')
print(f'Price columns: {len(price_cols)} - {price_cols}')
print(f'Return columns: {len(return_cols)} - {return_cols}')
print(f'Technical indicators: {len(tech_indicator_cols)}')
print(f'Global market features: {len(global_market_cols)}')
print(f'GDELT features: {len(gdelt_cols)}')
print(f'Macro features: {len(macro_cols)}')
print('-' * 60)

FEATURE CATEGORIES
------------------------------------------------------------
ID columns: 2 - ['Date', 'Ticker']
Price columns: 5 - ['Open', 'High', 'Low', 'Close', 'Volume']
Return columns: 2 - ['Return', 'Log_Return']
Technical indicators: 13
Global market features: 8
GDELT features: 7
Macro features: 4
------------------------------------------------------------


In [33]:
# Data leakage prevention confirmation
print('DATA LEAKAGE PREVENTION')
print('-' * 60)
print('The following features have been shifted by 1 day:')
print(f'  - Global market returns: {global_columns}')
print(f'  - GDELT event features: {gdelt_columns}')
print(f'  - Macro economic features: {macro_columns}')
print('This ensures the model only uses information available at time t to predict t+1.')
print('-' * 60)

DATA LEAKAGE PREVENTION
------------------------------------------------------------
The following features have been shifted by 1 day:
  - Global market returns: ['SP500_RET', 'NASDAQ_RET', 'DOW_RET', 'GOLD_RET', 'OIL_RET', 'USDINR_RET', 'VIX_RET', 'NIFTY_RET']
  - GDELT event features: ['Event_Count', 'Avg_Tone', 'War_Flag', 'Crisis_Flag', 'Inflation_Flag', 'Rate_Hike_Flag', 'Recession_Flag']
  - Macro economic features: ['Interest_Rate', 'Inflation', 'Unemployment', 'GDP']
This ensures the model only uses information available at time t to predict t+1.
------------------------------------------------------------


In [34]:
# Sample data for each ticker
print('SAMPLE DATA (First row per ticker)')
print('-' * 60)
sample_per_ticker = merged_df.groupby('Ticker').first().reset_index()
sample_per_ticker[['Ticker', 'Date', 'Close', 'Return', 'RSI', 'SP500_RET', 'Event_Count', 'Interest_Rate']].head(10)

SAMPLE DATA (First row per ticker)
------------------------------------------------------------


,Ticker,Date,Close,Return,RSI,SP500_RET,Event_Count,Interest_Rate
0,ABB,2023-03-15,3302.250000,-0.002688,61.089784,0.016477,1201.0,4.65
1,ADANIENSOL,2023-10-30,759.250000,-0.006867,42.058405,-0.004800,770.0,5.33
2,ADANIENT,2023-03-15,1839.000000,0.057991,46.872532,0.016477,1201.0,4.65
3,ADANIGREEN,2023-03-15,740.400024,0.048577,47.203111,0.016477,1201.0,4.65
4,ADANIPORTS,2023-03-15,679.349976,0.037968,57.676440,0.016477,1201.0,4.65
5,ADANIPOWER,2023-03-15,40.360001,-0.012479,56.692139,0.016477,1201.0,4.65
6,AMBUJACEM,2023-03-15,365.149994,0.033248,45.842050,0.016477,1201.0,4.65
7,APOLLOHOSP,2023-03-15,4328.799805,0.006791,44.154739,0.016477,1201.0,4.65
8,ASIANPAINT,2023-03-15,2827.399902,0.029793,51.598051,0.016477,1201.0,4.65
9,AXISBANK,2023-03-15,824.049988,-0.010566,34.932564,0.016477,1201.0,4.65


In [35]:
# Final confirmation
print('=' * 60)
print('STATUS: DATASET READY FOR FEATURE ENGINEERING')
print('=' * 60)
print('Next steps:')
print('1. Run data checks and validation (09_data_checks_and_validation.ipynb)')
print('2. Perform EDA (10_eda.ipynb)')
print('3. Create target variable and additional features (11_preprocessing_and_feature_engineering.ipynb)')
print('=' * 60)
print('\nNOTE: Target variable and lag features will be created in subsequent notebooks.')
print('This notebook is strictly for merging and alignment only.')

STATUS: DATASET READY FOR FEATURE ENGINEERING
Next steps:
1. Run data checks and validation (09_data_checks_and_validation.ipynb)
2. Perform EDA (10_eda.ipynb)
3. Create target variable and additional features (11_preprocessing_and_feature_engineering.ipynb)

NOTE: Target variable and lag features will be created in subsequent notebooks.
This notebook is strictly for merging and alignment only.
